# Example 9.6.1 / 2: Reshape Wide and Long Tables

Companion notebook for **9.6.1 Repeated Ingestion and Data Reshaping** in the AMPL lab Documentation, section "Reshape Wide and Long Tables". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Reshape Wide and Long Tables

Long form (first table) has one row per complete key; wide or matrix form (second table) has
one index down the rows and another across the columns:

| product | period | market |
|---|---:|---:|
| bands | 1 | 6000 |
| bands | 2 | 5000 |
| coils | 1 | 4000 |

| product | p1 | p2 |
|---|---:|---:|
| bands | 6000 | 5000 |
| coils | 4000 | 4500 |

A table declaration can read either form into the same two-index parameter. For the wide
form, the column-iteration form `{t in PERIOD} <...>` generates one column per period, with the
heading computed by a string expression. A second declaration writes the long form:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
set PERIOD;
param market {PROD, PERIOD} >= 0;

table Wide IN "amplcsv" "market_wide.csv":
    [product], {t in PERIOD} <market[product, t] ~ ("p" & t)>;
table Long OUT "amplcsv" "market_long.csv":
    [product, period], market;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
with open("market_wide.csv", "w") as f:
    f.write("product,p1,p2\nbands,6000,5000\ncoils,4000,4500\nplate,3500,3000\n")
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.set["PERIOD"] = [1, 2]

In [5]:
# run.py
ampl.eval("load amplcsv.dll;")
ampl.read_table("Wide")
ampl.write_table("Long")
print(open("market_long.csv").read())

product,period,market
bands,1,6000
bands,2,5000
coils,1,4000
coils,2,4500
plate,1,3500
plate,2,3000



### Pure AMPL: data.dat and model.run

Long form makes compound keys, duplicates and missing combinations explicit, so it is the
better exchange format; wide form is easier for people to edit. Converting at the table
declaration keeps the model indexed the same way whichever form the file uses.

Pure AMPL, with the same `model.mod` and `market_wide.csv`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
set PERIOD := 1 2;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
load amplcsv.dll;
read table Wide;
write table Long;
display market;

market :=
bands 1   6000
bands 2   5000
coils 1   4000
coils 2   4500
plate 1   3500
plate 2   3000
;

